In [ ]:
import keras
from keras.api.callbacks import EarlyStopping
from keras.api.layers import TextVectorization
import numpy as np
import pickle

# Datos de entrenamiento
texts = [
    "open browser", "open the browser", "open chrome", "search in browser", "search in chrome", "start chrome", "start the browser", "start browser", "launch chrome", "launch the browser", "launch browser", 
    "please open browser", "please open the browser", "please open chrome", "please search in browser", "please search in chrome", "please start chrome", "please start the browser", "please start browser", "please launch chrome", "please launch the browser", "please launch browser", 
    "close browser", "close the browser", "close chrome", "end chrome", "end browser", "end the browser", "shut chrome", "shut brower", "shut the browser",
    "please close browser", "please close the browser", "please close chrome", "please end chrome", "please end browser", "please end the browser", "please shut chrome", "please shut brower", "please shut the browser",
    "open vlc", "open music player", "open the music player", "reproduce music", "reproduce music in vlc", "turn the music on", "launch vlc", "launch music player", "launch the music player",
    "please open vlc", "please open music player", "please open the music player", "please reproduce music", "please reproduce music in vlc", "please turn the music on", "please launch vlc", "please launch music player", "please launch the music player",
    "close vlc", "close music player", "close the music player", "shut vlc", "shut music player", "shut the music player", "turn the music off", "end vlc", "end music player", "end the music player",
    "please close vlc", "please close music player", "please close the music player", "please shut vlc", "please shut music player", "please shut the music player", "please turn the music off", "please end vlc", "please end music player", "please end the music player",
    "open whatsapp", "open the whatsapp", "start whatsapp", "start the whatspp", "launch whatsapp", "launch the whatsapp", "lets chit chat", "lets chit chatting",
    "please open whatsapp", "please open the whatsapp", "please start whatsapp", "please start the whatspp", "please launch whatsapp", "please launch the whatsapp", 
    "close whatsapp", "close the whatsapp", "end whatsapp", "end the whatsapp", "shut whatsapp", "shut the whatsapp",
    "please close whatsapp", "please close the whatsapp", "please end whatsapp", "please end the whatsapp", "please shut whatsapp", "please shut the whatsapp",
]
labels = [
    "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser",
    "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser", "open_browser",
    "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser",
    "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser", "close_browser",
    "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player",
    "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player", "open_music_player",
    "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player",
    "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player", "close_music_player",
    "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp",
    "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp", "open_whatsapp",
    "close_whatsapp", "close_whatsapp", "close_whatsapp", "close_whatsapp", "close_whatsapp", "close_whatsapp",
    "close_whatsapp", "close_whatsapp", "close_whatsapp", "close_whatsapp", "close_whatsapp", "close_whatsapp",
]

# Mapeo de etiquetas
label_mapping = {label: i for i, label in enumerate(set(labels))}
y_train = np.array([label_mapping[label] for label in labels])

# TextVectorization en lugar de Tokenizer
vectorizer = TextVectorization(max_tokens=1000, output_mode='int', output_sequence_length=10)
vectorizer.adapt(texts)
padded_sequences = vectorizer(texts)

# Crear modelo
model = keras.Sequential([
    keras.layers.Embedding(input_dim=1000, output_dim=64),
    keras.layers.Bidirectional(keras.layers.LSTM(32, return_sequences=True)),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(label_mapping), activation='softmax')
])

# Callbacks y compilación
early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Entrenamiento
model.fit(padded_sequences, y_train, epochs=100, batch_size=4, callbacks=[early_stop])

# Guardar el modelo y el vectorizador
model.save("liz_0001.keras")
with open("vectorizer.pickle", "wb") as handle:
    pickle.dump(vectorizer, handle)
with open("label_mapping.pickle", "wb") as handle:
    pickle.dump(label_mapping, handle)

In [ ]:
model: keras.Sequential = keras.models.load_model("liz_0001.keras")
with open("vectorizer.pickle", "rb") as handle:
    vectorizer = pickle.load(handle)
with open("label_mapping.pickle", "rb") as handle:
    label_mapping: dict = pickle.load(handle)

In [7]:
import os

# Funciones para ejecutar comandos
def open_browser(): os.system("start chrome")
def close_browser(): os.system("taskkill /IM chrome.exe /F")
def open_music_player(): os.system("start vlc")
def close_music_player(): os.system("taskkill /IM vlc.exe /F")
def open_whatsapp(): os.system("start whatsapp")
def close_whatsapp(): os.system("taskkill /IM WhatsApp.exe /F")

actions = {
    "open_browser": open_browser,
    "close_browser": close_browser,
    "open_music_player": open_music_player,
    "close_music_player": close_music_player,
    "open_whatsapp": open_whatsapp,
    "close_whatsapp": close_whatsapp,
}

In [8]:
def get_intent(text):
    vectorized_input = vectorizer([text])
    prediction = model.predict(vectorized_input)
    intent = list(label_mapping.keys())[np.argmax(prediction)]
    return intent

In [9]:
import pyaudio
import vosk
import json

def escuchar():
    model = vosk.Model("./voice_engine/models/vosk-model-en-us-0.42-gigaspeech")
    recognizer = vosk.KaldiRecognizer(model, 16000)

    audio = pyaudio.PyAudio()
    stream = audio.open(format=pyaudio.paInt16, channels=1, rate=16000, input=True, frames_per_buffer=8192)
    stream.start_stream()

    print("🎤 Escuchando...")
    while True:
        data = stream.read(4096, exception_on_overflow=False)
        if recognizer.AcceptWaveform(data):
            result = json.loads(recognizer.Result())
            return result["text"]
        
def assistant():
    command = escuchar()
    print(f"👤 Tú: {command}")
    if command:
        intent = get_intent(command)
        print(f"🤖 Intención detectada: {intent}")
        actions.get(intent, lambda: print("No entiendo ese comando"))()

In [ ]:
assistant()